# Day 2 Lab - Groq + TinyLlama (Local WSL)

In [ ]:
# Cell 1: Imports and setup
import os
from dotenv import load_dotenv
from scraper import fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI
import requests

load_dotenv(override=True)
groq_api_key = os.getenv('GROQ_API_KEY')

if not groq_api_key:
    print("No Groq API key was found - please add GROQ_API_KEY to your .env file")
elif not groq_api_key.startswith("gsk_"):
    print("An API key was found, but it doesn't start with 'gsk_' - please check you're using the right Groq API key")
else:
    print("Groq API key found and looks good so far!")

Groq API key found and looks good so far!


In [ ]:
# Cell 2: Configure Groq client
groq_client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=groq_api_key
)

In [ ]:
# Cell 3: Test Groq connection
response = groq_client.chat.completions.create(
    model="openai/gpt-oss-120b", 
    messages=[{"role": "user", "content": "Tell me a fun fact about coding"}]
)
print(response.choices[0].message.content)

Here’s a quirky tidbit that many programmers love to quote:

**The very first “computer program” was written for a machine that didn’t even have a screen, a keyboard, or a mouse.**

In 1842–1843, mathematician **Ada Lovelace** wrote what we now recognize as the first algorithm—an instruction set for Charles Babbage’s *Analytical Engine*, a mechanical “computer” that was never completed. Her notes included a step‑by‑step method for calculating Bernoulli numbers, and she even anticipated the idea of looping and conditional logic—concepts that are core to modern programming languages today.

So, the next time you type `print("Hello, world!")`, remember you’re standing on the shoulders of a 19th‑century visionary who was telling a giant brass gear machine how to think! 🚀


In [ ]:
# Cell 4: Test other Groq models
models = ["moonshotai/kimi-k2-instruct-0905", "qwen/qwen3-32b"]
for model in models:
    print(f"\nTesting {model}:")
    response = groq_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": "Tell me a fun fact  about coding"}]
    )
    print(response.choices[0].message.content)


Testing moonshotai/kimi-k2-instruct-0905:
The first computer bug was an actual bug: in 1947, Grace Hopper found a moth trapped in the Harvard Mark II’s relay, taped it in the logbook, and wrote “First actual case of bug being found.”

Testing qwen/qwen3-32b:
<think>
Okay, the user asked for a fun fact about coding. Let me think about what's interesting here. First, I should recall some unique aspects of coding. Oh right, did you know that the first computer bug was an actual moth? That's a classic one. The story goes that in 1947, a moth got stuck in the Harvard Mark II computer's relay, and Grace Hopper documented it and coined the term "debugging" to remove the insect. That's pretty cool. It's a historical tidbit and shows how early computing had real bugs.

Wait, but maybe the user wants something more recent or technical? Let me consider other facts. There's also the fact that the first program was written by Ada Lovelace, who wrote an algorithm for Charles Babbage's Analytical En

In [ ]:
# Cell 5: Setup local TinyLlama in WSL
ollama_url = "http://localhost:11434"

try:
    response = requests.get(ollama_url, timeout=2)
    if response.status_code == 200:
        print("✅ Ollama is running locally")
        
        models_response = requests.get(f"{ollama_url}/api/tags", timeout=2)
        if models_response.status_code == 200:
            models = models_response.json().get('models', [])
            print("Available models:")
            for model in models:
                print(f"  - {model['name']}")
            
            if any('tinyllama' in m['name'] for m in models):
                print("✅ tinyllama is available")
            else:
                print("❌ tinyllama not found. Run: ollama pull tinyllama")
    else:
        print("❌ Ollama returned unexpected status")
except Exception as e:
    print(f"❌ Ollama not running. Start Ollama first with: ollama serve")

✅ Ollama is running locally
Available models:
  - tinyllama:1.1b
✅ tinyllama is available


In [ ]:
# Cell 6: Test TinyLlama
try:
    response = requests.post(
        f"{ollama_url}/api/chat",
        json={
            "model": "tinyllama:1.1b",
            "messages": [{"role": "user", "content": "Tell me a fun fact"}],
            "stream": False
        },
        timeout=20
    )
    if response.status_code == 200:
        print("✅ TinyLlama is working")
        print(response.json()['message']['content'])
    else:
        print(f"Error: {response.status_code}")
except Exception as e:
    print(f"Error: {e}")

✅ TinyLlama is working
Sure, here's a fun fact for you:

Did you know that the word "fact" comes from the French word "faits" which means "made"? The first record of the word "fact" is from the 12th century, and it wasn't used as a noun until the 19th century.

So if you hear someone saying "I made a fact," it means that they've made something up!


## Website Summarization Exercise

In [ ]:
# Cell 7: System prompts
system_prompt = """
You are a helpful assistant that analyzes the contents of a website,
and provides a short summary, ignoring text that might be navigation related.
Respond in markdown.
"""

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]

In [ ]:
# Cell 8: Summary functions
def summarize_with_groq(url, model="openai/gpt-oss-120b"):
    website = fetch_website_contents(url)
    response = groq_client.chat.completions.create(
        model=model,
        messages=messages_for(website)
    )
    return response.choices[0].message.content

def summarize_with_tinyllama(url):
    website = fetch_website_contents(url)
    response = requests.post(
        f"{ollama_url}/api/chat",
        json={
            "model": "tinyllama:1.1b",
            "messages": messages_for(website),
            "stream": False
        },
        timeout=120
    )
    return response.json()['message']['content']

def display_summary(url, use_local=False, groq_model="openai/gpt-oss-120b"):
    if use_local:
        summary = summarize_with_tinyllama(url)
        print(f"\nSummary from TinyLlama for {url}:\n")
    else:
        summary = summarize_with_groq(url, groq_model)
        print(f"\nSummary from Groq {groq_model} for {url}:\n")
    
    display(Markdown(summary))

In [ ]:
# Cell 9: Test with Groq models
display_summary("https://jay-kalsariya.vercel.app", groq_model="openai/gpt-oss-120b")
display_summary("https://jay-kalsariya.vercel.app", groq_model="moonshotai/kimi-k2-instruct-0905")
display_summary("https://jay-kalsariya.vercel.app", groq_model="qwen/qwen3-32b")


Summary from Groq openai/gpt-oss-120b for https://jay-kalsariya.vercel.app:



**Jay Kalsariya – Portfolio Summary**

- **Who:** Jay Kalsariya, a Computer Engineering student at MBIT (CVM University) in Anand, India.  
- **Focus:** Full‑stack web development and AI/ML solutions.  

### Key Skills
- **Languages:** Java, Python, JavaScript, HTML, CSS  
- **Frontend:** React.js, HTML5, CSS3, ES6+  
- **Backend:** Node.js, Express.js, Java  
- **Databases:** MySQL, PostgreSQL, MongoDB  
- **AI/ML:** TensorFlow, neural networks, data analysis  

### Education
- B.Tech. Computer Engineering (2022‑2026)  
- Minor in Robotics (2023‑2025)  

### Certifications & Achievements
- Google Developer Student Club – Google Cloud Study Jams & Generative AI certification  
- Coursera Deep Learning Specialization  
- Python for Data Science certification  
- **Hackathon:** Top‑10 finish (AI‑driven solution) among 100+ teams at DAIICT Hackathon  
- SAP Foundation – Code Unnati Program & Innovation Marathon semifinalist (GTU)  

### Interests
- Hackathons, chess, paradoxology, cricket  

### Contact
- Simple contact form (Name, Email, Message) for inquiries.  

**Overall:** The site showcases Jay’s technical expertise, academic background, notable certifications, and competition successes, positioning him as a capable full‑stack developer with a strong AI/ML focus. No recent news or announcements beyond the listed achievements.


Summary from Groq moonshotai/kimi-k2-instruct-0905 for https://jay-kalsariya.vercel.app:



Jay Kalsariya is a Computer-Engineering student at MBIT, Anand (graduating 2026) and full-stack developer who builds responsive web apps and AI models.  
Core skills: React.js, Node.js, Python, Java, TensorFlow, SQL/NoSQL databases.  
Notable achievements: top-10 finish at DAIICT Hackathon, Google Cloud & Generative-AI certifications, Deep-Learning Specialization, Code Unnati semifinalist.


Summary from Groq qwen/qwen3-32b for https://jay-kalsariya.vercel.app:



<think>
Okay, let me start by reading through the content provided. The user wants a summary of a website belonging to Jay Kalsariya. First, I'll identify the main sections: a header with some cryptic terms like Neural Frequency Matched and Quantum Key Verified, followed by standard navigation links. Then there's the main content.

Under "About Me," Jay is a Computer Engineering student at MBIT, with full-stack web development and AI expertise. He lists his education, skills in various programming languages, frontend and backend technologies, databases, AI frameworks. Then certifications and achievements include Google Developer Club, Coursera courses, hackathon positions, and SAP programs. Projects and contact info are mentioned, along with some footers.

I need to ignore navigation elements like Home, About, etc. The main summary should highlight his role, education, skills, certifications, projects, and maybe contact. Since the projects section is listed with a call to contact, but there's no detailed project description here, I should note that he has a projects section but not describe them as they are not detailed in the content.

Also, check for any news or announcements. The content doesn't mention any recent news, just his profile and certifications. So the summary should focus on his background, skills, and achievements. Make sure it's concise, bullet points might help, but since the user wants a markdown response without using markdown in the thinking process, but the final answer will have markdown. Wait, the user said to respond in markdown. The initial instruction said to ignore text that might be navigation-related. The header with Neural Frequency Matched and Quantum Key Verified seems like a personal branding or website design element, but since it's not part of the main content, perhaps I can mention it in the summary as part of the unique website design. However, the main focus is on his role, education, skills, certifications, and contact info.

I should ensure that certifications and achievements are included. The footer has a copyright notice and navigation links again. The summary should be short, so I'll condense the info, perhaps starting with his current role, education, key skills, notable certifications, and a mention of contact. Avoid mentioning the navigation links unless relevant. Since there's no news or announcements, just the regular website structure, the summary doesn't need a section for that. The projects are listed but without details, so just note that he has a projects section.
</think>

Here's a summary of the Jay Kalsariya portfolio website:  

**Overview**  
Jay Kalsariya is a **Computer Engineering student** at MBIT, Anand, with a focus on **full-stack web development** and **artificial intelligence**. He positions himself as a technologist blending software development, AI, and robotics.  

**Key Sections**  
- **Education**: Bachelor of Technology in Computer Engineering at MBIT, with a robotics minor.  
- **Skills**:  
  - Frontend: React.js, HTML5, CSS3, JavaScript.  
  - Backend: Node.js, Express.js, Java.  
  - Databases: MySQL, PostgreSQL, MongoDB.  
  - AI/ML: TensorFlow, neural networks, data analysis.  
- **Certifications/Achievements**:  
  - Google Cloud Study Jams and Generative AI certifications.  
  - Coursera Deep Learning Specialization.  
  - Top 10 in AI-driven solution at DAIICT Hackathon.  
  - SAP Code Unnati Program finalist.  
- **Interests**: Hackathons, chess, cricket.  

**Design & Navigation**  
The site features a futuristic interface with terms like "Neural Frequency Matched" and "Quantum Key Verified," reflecting his focus on advanced technology. Navigation includes standard sections (Home, About, Projects, Contact).  

**Contact**  
A contact form is available, though no active news or recent project updates are detailed.  

The website emphasizes Jay's technical expertise, academic background, and achievements, framed within a unique sci-fi-inspired design.

In [ ]:
# Cell 10: Test with TinyLlama
display_summary("https://edwarddonner.com", use_local=True)


Summary from TinyLlama for https://edwarddonner.com:



Here is a short summary of the contents of the website:

Home - Edward Donner
AI Curriculuum
Proficient AI Engineer
Connect Four
Outsmart
An arenas that pit LLMs against each other in a battle of diplomacy and deviousness.

About
Posts
Well, hi there. I’m Ed. I’m a co-founder and CTО of Nebula.io, applying AI to a field where it can make a massive, positive impact: helping people discover their potential and pursue their reason for being. I’m previously the founder and CEO of a startup called utapt, acquired in 2021. I’m super grateful for your interest in my courses, which have been rated among the best in the industry. To my total joy, you’ll find my course offerings here. If you’re visiting from one of my courses, I’ll be grateful for every email and hope to add value with every message. I’ll keep you updated with occasional newsletters.

Keep in touch
Submitting form

Enter your email...
Stay in touch
Submitting form
Delta February 17, 2026
Vibe Coder to Agentic Engineer – Resources

AI Builders with n8n – Create Agent and Voice Agent Resources
November 11, 2025
AI Live Events – Links to Enroll in Production AI
September 15, 2025
AI Engineering MLOpd Track – Deploy AI to Production Resources
Navigate

In [ ]:
# Cell 11: Try different websites
websites = ["https://cnn.com", "https://anthropic.com"]
for url in websites:
    print(f"\n{'='*60}")
    print(f"Summarizing: {url}")
    print('='*60)
    display_summary(url, groq_model="qwen/qwen3-32b")


Summarizing: https://cnn.com

Summary from Groq qwen/qwen3-32b for https://cnn.com:



<think>
Okay, I need to summarize the contents of a website based on the provided HTML and text. The user mentioned to ignore navigation-related text and focus on the main content. Let me look through the content given.

First, the main title is "Breaking News, Latest News and Videos | CNN". CNN is a news organization, so this is the home page likely displaying recent news and sections. There's a form for ad feedback asking about relevance and technical issues. Then, various sections like US, World, Politics, Business, etc., are listed multiple times—probably navigation menus. There are also links to "Ukraine-Russia War" and "Israel-Hamas War," suggesting these are current major topics covered. The page includes login and subscription options. The bottom sections have links to different categories and features like Weather, Climate, Science, which are common news sections.

The user wants a summary ignoring navigation text. The main content seems to be the CNN homepage offering global news, videos, and various categories. The news sections mentioned include current conflicts, which are highlighted. There's no actual news content here, just sections and links, so the summary should focus on the structure and purpose of the site as a news source with these categories. I should mention the key sections and note the emphasis on live news and global events like the mentioned wars. Technical aspects like the feedback section can be omitted unless relevant. Avoid listing every link but note the main features. Also, no specific news articles are present, just the structure. The summary should state it's CNN's homepage providing global news across multiple topics, with a focus on current events and multimedia content.
</think>

The website is the homepage of **CNN** (Cable News Network), a major global news organization. It focuses on delivering **breaking news, latest updates, and videos** across various categories such as politics, business, health, and world events. Key highlights include:

- **Current conflict coverage**: Sections for the **Ukraine-Russia War** and **Israel-Hamas War** indicate ongoing coverage of major global crises.
- **Multimedia content**: Options to watch videos, live TV, and listen to audio streams, emphasizing multimedia delivery.
- **Interactive elements**: A user feedback form for ad performance (e.g., technical issues or relevance) and login/sign-up functionality for personalized accounts.
- **Extensive categories**: Topics like Science, Climate, Technology, and Sports are grouped under a global news framework.

The site serves as a hub for real-time news, with a navigation-focused layout that prioritizes user engagement through subscriptions, regional editions, and diverse content streams. No specific news articles are present in the provided content, but the structure reflects CNN’s role as a comprehensive news platform.


Summarizing: https://anthropic.com

Summary from Groq qwen/qwen3-32b for https://anthropic.com:



<think>
Okay, let's tackle this. The user provided the contents of a website and wants a summary, including any news or announcements. First, I need to parse through the given content to identify the main sections and key information.

Looking at the text, the main sections are "Research," "Learn," "Company," "Try Claude," "Products," and "Models." There's a lot of repetition of navigation links like "Log in to Claude" and "Download app," which are probably part of the website's menu and should be ignored as per the user's instruction to ignore navigation-related text.

Under the main heading, there's a statement about Anthropic being a public benefit corporation focused on AI safety. That's important. Then there's a featured section about "Four Hundred Meters on Mars," which mentions an AI-planned drive on another planet. That seems like a project highlight.

The latest releases section has two items. One is "Where things stand with the Department of War" by Dario Amodei, dated March 5, 2026. The other is "Claude Sonnet 4.6" with a date of February 17, 2026. These are news/announcements. The Sonnet 4.6 announcement mentions it's the most capable model yet with improvements in coding, agents, and professional work.

I need to summarize Anthropic's mission, mention the key products like Claude and its variants (Code, Platform), and the models (Opus, Sonnet, Haiku). Then, highlight the two latest announcements: the Department of War post and the new Sonnet model. Also, mention the featured project on Mars. Avoid including the repetitive navigation items and focus on the unique content.

Wait, the dates given are into the future (2026). That might be a typo or hypothetical scenario. The user probably wants the summary as presented, so I'll note the dates as they are. Also, ensure the summary is concise and in markdown, as instructed. Ignore any login/download app links that are part of the navigation.
</think>

**Summary of Anthropic's Website**

Anthropic is a public benefit corporation focused on advancing AI research and products with safety as a core priority. The company emphasizes securing AI's benefits while mitigating risks through initiatives like **Claude's Constitution**, **Responsible Scaling Policy**, and **Transparency**.

### Key Highlights:
- **Products**:  
  - **Claude** (Chat model, Code model, Platform).  
  - **Models**: Opus, Sonnet, and Haiku cater to diverse professional and coding tasks.
- **Research & Innovation**:  
  - Featured project: *"Four Hundred Meters on Mars"* — the first AI-planned drive on another planet.
  - **Claude Sonnet 4.6** (Feb 17, 2026): Introduced as the most capable Sonnet model with improved performance for coding, agents, and professional workflows.

### Recent News/Announcements:
1. **March 5, 2026**: *Where things stand with the Department of War* — a statement by CEO Dario Amodei addressing AI safety and policy concerns.
2. **February 17, 2026**: Launch of **Claude Sonnet 4.6**, highlighting enhanced capabilities across key domains.

### Additional Resources:
- **Anthropic Academy**: Tutorials, use cases, and developer documentation.
- **Transparency & Compliance**: Security certifications and responsible AI practices.
- **Careers & Events**: Opportunities and updates on AI advancements.

The site promotes Anthropic's mission to lead ethical AI development, balancing innovation with societal impact.